# Busca Semântica com Álgebra Linear e AWS S3

**Autor:** Rafael  
**Contexto:** Projeto de portfólio — MLOps / GenAI  

---

## O que vamos construir?

Um motor de busca que entende o **significado** das palavras, não apenas a presença delas.  
Quando você digitar *"como monitorar modelos em produção"*, ele vai encontrar documentos sobre **CloudWatch e MLflow** — mesmo que a frase exata não apareça em nenhum deles.

## Por que isso importa?

Esse mecanismo é a base de:
- **RAG (Retrieval-Augmented Generation)** — como o ChatGPT busca contexto relevante
- **Sistemas de recomendação** — Netflix, Spotify, Amazon
- **Buscas inteligentes** em bases de conhecimento corporativas

## Conceitos de álgebra linear que vamos usar

| Conceito | Onde aparece |
|---|---|
| Norma de vetor | Normalizar tamanho dos documentos |
| Produto escalar | Medir quanto dois vetores se alinham |
| Similaridade de cosseno | Ranquear resultados da busca |

## Parte 1 — Instalação e imports

In [ ]:
# Instala dependências (execute apenas na primeira vez)
!pip install -q numpy scikit-learn boto3

In [ ]:
import json
import sys
import numpy as np
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer

# Adiciona o diretório src ao path para importar nosso módulo
sys.path.append("../src")
from vector_search import norma_vetor, produto_escalar, similaridade_cosseno, MotorBuscaSemantica

print("Imports OK!")

---
## Parte 2 — Álgebra Linear do zero

Antes de usar ferramentas prontas, vamos entender o que acontece por baixo.  
Cada documento vai ser representado como um **ponto num espaço de alta dimensão**.  
Quanto mais próximos dois pontos estiverem, mais similares são os documentos.

In [ ]:
# --- NORMA DE VETOR ---
# A norma mede o "tamanho" de um vetor.
# Em busca semântica: usamos para normalizar documentos longos e curtos na mesma escala.

doc_curto  = np.array([1, 0, 1])   # documento com 2 palavras relevantes
doc_longo  = np.array([3, 0, 3])   # mesmo conteúdo, mas texto 3x mais longo

print(f"Norma doc_curto : {norma_vetor(doc_curto):.4f}")
print(f"Norma doc_longo : {norma_vetor(doc_longo):.4f}")
print()
print("Sem normalização, doc_longo parece 'mais relevante' só por ser maior.")
print("A similaridade de cosseno resolve isso dividindo pela norma.")

In [ ]:
# --- PRODUTO ESCALAR ---
# Multiplica elemento a elemento e soma. Mede o quanto dois vetores "apontam juntos".
# Aplicação: base do cálculo de similaridade entre query e documento.

query = np.array([1, 0, 1, 0])       # busca por "machine learning deploy"
doc_a = np.array([1, 0, 1, 0])       # documento sobre ML e deploy
doc_b = np.array([0, 1, 0, 1])       # documento sobre assunto diferente

print(f"Produto escalar (query · doc_a): {produto_escalar(query, doc_a):.1f}  ← alta similaridade")
print(f"Produto escalar (query · doc_b): {produto_escalar(query, doc_b):.1f}  ← nenhuma relação")

In [ ]:
# --- SIMILARIDADE DE COSSENO ---
# É o produto escalar dividido pelo produto das normas.
# Resultado entre -1 e 1: quanto mais próximo de 1, mais similar.

pares = [
    ("idênticos",    np.array([1, 1, 0]), np.array([1, 1, 0])),
    ("similares",    np.array([1, 1, 0]), np.array([1, 0.8, 0.2])),
    ("perpendicular",np.array([1, 0, 0]), np.array([0, 1, 0])),
]

print(f"{'Par':<15} {'Similaridade':>12}")
print("-" * 30)
for nome, va, vb in pares:
    sim = similaridade_cosseno(va, vb)
    print(f"{nome:<15} {sim:>12.4f}")

In [ ]:
# --- VISUALIZAÇÃO: por que cosseno e não distância euclidiana? ---
# O ângulo entre os vetores não muda com o tamanho — por isso a similaridade
# de cosseno é robusta a textos de tamanhos diferentes.

fig, ax = plt.subplots(figsize=(5, 5))

vetores = {
    "doc_curto [1,1]":  np.array([1, 1]),
    "doc_longo [3,3]":  np.array([3, 3]),
    "doc_diferente [1,0]": np.array([1, 0]),
}
cores = ["royalblue", "cornflowerblue", "tomato"]

for (nome, v), cor in zip(vetores.items(), cores):
    ax.annotate("", xy=v, xytext=(0, 0),
                arrowprops=dict(arrowstyle="->", color=cor, lw=2))
    ax.text(v[0] + 0.05, v[1] + 0.05, nome, color=cor, fontsize=9)

sim_cc = similaridade_cosseno(np.array([1,1]), np.array([3,3]))
ax.set_title(f"doc_curto vs doc_longo: similaridade = {sim_cc:.2f}\n(mesmo ângulo = mesma similaridade)",
             fontsize=10)
ax.set_xlim(-0.2, 4); ax.set_ylim(-0.2, 4)
ax.axhline(0, color="gray", lw=0.5); ax.axvline(0, color="gray", lw=0.5)
ax.set_aspect("equal")
plt.tight_layout()
plt.show()

---
## Parte 3 — Motor de busca real

Agora aplicamos os conceitos num sistema completo.  
Temos 8 documentos sobre tecnologias de Cloud e MLOps.  
O motor vai encontrar os mais relevantes para qualquer query.

In [ ]:
# Inicializa o motor e indexa os documentos
motor = MotorBuscaSemantica()
motor.carregar_documentos("../data/documentos.json")

print(f"\nVocabulário TF-IDF: {motor.matriz_tfidf.shape[1]} dimensões")
print(f"Cada documento virou um vetor de {motor.matriz_tfidf.shape[1]} números.")

In [ ]:
# Teste 1: busca por conceito de MLOps
query = "como monitorar modelos em produção"
resultados = motor.buscar(query, top_k=3)

print(f"Query: '{query}'")
print("=" * 60)
for i, r in enumerate(resultados, 1):
    print(f"\n#{i} — Score: {r['score']}")
    print(f"   {r['titulo']}")
    print(f"   {r['texto'][:80]}...")

In [ ]:
# Teste 2: busca por conceito de deploy
query = "empacotar e fazer deploy de modelo"
resultados = motor.buscar(query, top_k=3)

print(f"Query: '{query}'")
print("=" * 60)
for i, r in enumerate(resultados, 1):
    print(f"\n#{i} — Score: {r['score']}")
    print(f"   {r['titulo']}")

In [ ]:
# Teste 3 — experimente sua própria busca!
query = "armazenamento de datasets na nuvem"   # <- mude aqui
resultados = motor.buscar(query, top_k=3)

print(f"Query: '{query}'")
print("=" * 60)
for i, r in enumerate(resultados, 1):
    print(f"\n#{i} — Score: {r['score']}")
    print(f"   {r['titulo']}")

---
## Parte 4 — Integração com AWS S3

Em produção, recalcular o índice TF-IDF toda vez que o sistema inicia é custoso.  
A solução: salvar a matriz de vetores no S3 e carregá-la quando necessário.

> **Pré-requisito:** Configure suas credenciais AWS com `aws configure` antes de rodar.

In [ ]:
from vector_search import salvar_indice_s3, carregar_indice_s3

# --- Configure seu bucket aqui ---
BUCKET = "seu-bucket-mlops"   # <- substitua pelo nome do seu bucket
CHAVE  = "indices/tfidf_documentos.npy"

# Salva o índice vetorial no S3
sucesso = salvar_indice_s3(motor.matriz_tfidf, BUCKET, CHAVE)

if sucesso:
    print(f"Índice salvo em s3://{BUCKET}/{CHAVE}")
    print(f"Shape: {motor.matriz_tfidf.shape} — {motor.matriz_tfidf.nbytes / 1024:.1f} KB")

In [ ]:
# Carrega o índice do S3 e usa para busca — simulando um worker de produção
matriz_carregada = carregar_indice_s3(BUCKET, CHAVE)

if matriz_carregada is not None:
    print(f"Índice carregado — shape: {matriz_carregada.shape}")
    print("Pronto para servir requisições de busca!")

---
## Resumo e próximos passos

### O que construímos
- Implementação de norma, produto escalar e similaridade de cosseno do zero
- Motor de busca semântica com TF-IDF e ranking por similaridade
- Persistência do índice vetorial no Amazon S3

### Próximos passos para evoluir este projeto

1. **Substituir TF-IDF por embeddings reais**  
   Usar AWS Bedrock (modelo Titan Embeddings) para vetores muito mais ricos semanticamente

2. **Deploy como API**  
   Empacotar em Docker e fazer deploy no AWS Lambda + API Gateway

3. **Escalar o índice**  
   Migrar para Amazon OpenSearch — que é exatamente o que empresas usam em produção

4. **Pipeline automático**  
   Quando novos documentos chegam no S3, acionar Lambda para reindexar automaticamente